# Step 5: Building the Retriever Pipeline

This notebook completes Step 5 of the RAG guideline by building and validating a retriever pipeline on **Vertex AI Vector Search** for the Household Energy Optimization & Sustainability Q&A Support System.

Scope:
- Use `energy_for_rag.md` as the primary dataset.
- Generate embeddings with `intfloat/multilingual-e5-large`.
- Prepare and optionally upload embeddings to GCS.
- Create, deploy, and query a Vertex AI Vector Search index when flags are enabled.
- Implement dense retrieval, keyword-aware hybrid retrieval, reranking, and metadata filtering demonstrations.
- Produce a Step 5 deliverable: test results, comparison notes, and a retrieval recommendation.

Notes:
- This notebook is the main implementation path for Step 5.
- Vertex AI Vector Search is the primary retrieval backend, while local keyword scoring is used to demonstrate hybrid retrieval and reranking behavior.


## Cell Guide: Install Dependencies

This cell installs the Python packages required for Vertex AI, Cloud Storage, embedding generation, and notebook-based data processing.


In [1]:
!pip install -q google-cloud-aiplatform google-cloud-storage python-dotenv langchain-text-splitters langchain-huggingface sentence-transformers pandas tqdm

## Cell Guide: Load Libraries and Environment

This cell imports the required libraries, loads environment variables from `.env`, builds the runtime configuration, and initializes Vertex AI for the selected project and region.


In [12]:
import json
import os
import re
from datetime import UTC, datetime
from pathlib import Path
from time import sleep

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from google.api_core.exceptions import AlreadyExists, GoogleAPICallError, NotFound, ServiceUnavailable
from google.auth.exceptions import DefaultCredentialsError
from google.cloud import aiplatform, storage
from google.cloud.aiplatform.matching_engine import matching_engine_index_config as me_config
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

load_dotenv(find_dotenv(usecwd=True), override=False)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent

REQUIRED_ENV_VARS = [
    "PROJECT_ID",
    "REGION",
    "DEPLOYED_INDEX_ID",
    "EMBEDDING_MODEL_NAME",
]

missing = [key for key in REQUIRED_ENV_VARS if not os.getenv(key)]
if os.getenv("CREATE_VERTEX_RESOURCES", "false").lower() == "true" and not os.getenv("BUCKET_NAME"):
    missing.append("BUCKET_NAME")
if missing:
    raise ValueError(
        "Missing required environment variables: "
        + ", ".join(missing)
        + ". Update your .env from .env.example."
    )

CONFIG = {key: os.getenv(key) for key in REQUIRED_ENV_VARS}
CONFIG["BUCKET_NAME"] = os.getenv("BUCKET_NAME", "")
CONFIG["VERTEX_INDEX_DISPLAY_NAME"] = os.getenv("VERTEX_INDEX_DISPLAY_NAME", "rag-step4-energy-vector-index")
CONFIG["VERTEX_ENDPOINT_DISPLAY_NAME"] = os.getenv("VERTEX_ENDPOINT_DISPLAY_NAME", "rag-step4-energy-vector-endpoint")
CONFIG["CREATE_VERTEX_RESOURCES"] = os.getenv("CREATE_VERTEX_RESOURCES", "false").lower() == "true"
CONFIG["DEPLOY_VERTEX_INDEX"] = os.getenv("DEPLOY_VERTEX_INDEX", "false").lower() == "true"
CONFIG["RUN_VERTEX_QUERY"] = os.getenv("RUN_VERTEX_QUERY", "false").lower() == "true"
CONFIG["DELETE_VERTEX_RESOURCES"] = os.getenv("DELETE_VERTEX_RESOURCES", "false").lower() == "true"
CONFIG["VERTEX_INDEX_ID"] = os.getenv("VERTEX_INDEX_ID", "")
CONFIG["VERTEX_ENDPOINT_ID"] = os.getenv("VERTEX_ENDPOINT_ID", "")
CONFIG["VERTEX_INDEX_RESOURCE_NAME"] = os.getenv("VERTEX_INDEX_RESOURCE_NAME", "")
CONFIG["VERTEX_ENDPOINT_RESOURCE_NAME"] = os.getenv("VERTEX_ENDPOINT_RESOURCE_NAME", "")

RUN_STATE = {}


def mark_step(step: str, status: str, detail: str) -> None:
    RUN_STATE[step] = {
        "status": status,
        "detail": detail,
        "timestamp": datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z"),
    }


def auth_help() -> str:
    return (
        "Google Cloud authentication is required. Run:\n"
        "gcloud auth application-default login\n"
        "gcloud config set project rag-project-495601"
    )


try:
    aiplatform.init(project=CONFIG["PROJECT_ID"], location=CONFIG["REGION"])
except DefaultCredentialsError as exc:
    raise RuntimeError(auth_help()) from exc

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ID:", CONFIG["PROJECT_ID"])
print("REGION:", CONFIG["REGION"])
print("BUCKET_NAME:", CONFIG["BUCKET_NAME"] or "<not set>")
print("VERTEX_INDEX_ID:", CONFIG["VERTEX_INDEX_ID"] or "<not set>")
print("VERTEX_ENDPOINT_ID:", CONFIG["VERTEX_ENDPOINT_ID"] or "<not set>")
print("CREATE_VERTEX_RESOURCES:", CONFIG["CREATE_VERTEX_RESOURCES"])
print("DEPLOY_VERTEX_INDEX:", CONFIG["DEPLOY_VERTEX_INDEX"])
print("RUN_VERTEX_QUERY:", CONFIG["RUN_VERTEX_QUERY"])
mark_step("environment_loaded", "passed", "Environment variables loaded and Vertex AI initialized.")

PROJECT_ROOT: D:\Intern
PROJECT_ID: rag-project-495601
REGION: asia-southeast1
BUCKET_NAME: rag-bucket-xuandong
VERTEX_INDEX_ID: <not set>
VERTEX_ENDPOINT_ID: <not set>
CREATE_VERTEX_RESOURCES: True
DEPLOY_VERTEX_INDEX: True
RUN_VERTEX_QUERY: True


## Cell Guide: Load Source Data

This cell locates `energy_for_rag.md`, reads the full markdown content, and confirms the input document used in the Vector Search pipeline.


In [13]:
def find_energy_markdown(project_root: Path) -> Path:
    candidates = sorted(
        project_root.rglob("energy_for_rag.md"),
        key=lambda path: (len(path.relative_to(project_root).parts), str(path).lower()),
    )
    if not candidates:
        raise FileNotFoundError(
            "File energy_for_rag.md not found. Please place it in the project root or data directory."
        )
    chosen = candidates[0]
    if len(candidates) > 1:
        print("Multiple energy_for_rag.md files found:")
        for candidate in candidates:
            print(" -", candidate)
    print("Using data file:", chosen)
    return chosen


DATA_FILE = find_energy_markdown(PROJECT_ROOT)
raw_markdown = DATA_FILE.read_text(encoding="utf-8")
print("Characters:", len(raw_markdown))
print(raw_markdown[:1000])
mark_step("data_loaded", "passed", f"Loaded {DATA_FILE}.")

Using data file: D:\Intern\data\outputs\energy_for_rag.md
Characters: 12261
# Document: Household Energy Cost Analysis
# Source: AusNet

## Page 1

Household Energy
Cost Analysis
31 Jan 2025
Assumptions Document

---

## Page 2

Purpose Methodology Acquisition costs
Purpose and contents
Purpose Contents
•
Incorporates our current EDPR expenditure forecasts (i.e. the distribution component),
and relies on several internal (e.g. 2023 customer segmentation) and external, publicly
1. Purpose
available data sources (ECA ‘Stepping Up’ report) for other bill components and key
inputs/ assumptions 2. Methodology &
•
Captures annual household ‘running costs’ and the cost of solar PV and wiring if assumptions
applicable (i.e. upfront purchase costs of electrical appliances and EVs are excluded
3. Acquisition Costs
on the basis that these purchases will be made at end of life of existing
appliances/vehicle, at cost parity)
•
Includes the following three different household types, with and without

## Cell Guide: Split Into Chunks

This cell splits the source document into smaller chunks so they can be embedded, indexed, retrieved, and mapped back to readable source text.


In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n## ", "\n### ", "\n", ". ", " ", ""],
)

documents = splitter.create_documents(
    [raw_markdown],
    metadatas=[{"source_file": DATA_FILE.name, "source_path": str(DATA_FILE)}],
)


def infer_page_number(text):
    match = re.search(r"## Page\s+(\d+)", text)
    return int(match.group(1)) if match else None


def infer_section_tag(text):
    lowered = text.lower()
    if "assumption" in lowered:
        return "assumptions"
    if "methodology" in lowered:
        return "methodology"
    if "acquisition cost" in lowered:
        return "acquisition_costs"
    if "purpose" in lowered:
        return "purpose"
    return "general"


chunk_rows = []
for index, document in enumerate(documents):
    chunk_rows.append(
        {
            "chunk_id": f"energy_chunk_{index:04d}",
            "content": document.page_content,
            "char_count": len(document.page_content),
            "source_file": document.metadata["source_file"],
            "source_path": document.metadata["source_path"],
            "page_number": infer_page_number(document.page_content),
            "section_tag": infer_section_tag(document.page_content),
        }
    )

chunk_df = pd.DataFrame(chunk_rows)
chunk_df["page_number"] = chunk_df["page_number"].ffill().astype("Int64")
print("Chunk count:", len(chunk_df))
display(chunk_df.head(3))
mark_step("chunking", "passed", f"Created {len(chunk_df)} chunks from energy_for_rag.md with page and section metadata.")


Chunk count: 23


,chunk_id,content,char_count,source_file,source_path,page_number,section_tag
0,energy_chunk_0000,# Document: Household Energy Cost Analysis\n# ...,140,energy_for_rag.md,D:\Intern\data\outputs\energy_for_rag.md,1,assumptions
1,energy_chunk_0001,## Page 2\n\nPurpose Methodology Acquisition c...,791,energy_for_rag.md,D:\Intern\data\outputs\energy_for_rag.md,2,assumptions
2,energy_chunk_0002,"appliances/vehicle, at cost parity)\n•\nInclud...",259,energy_for_rag.md,D:\Intern\data\outputs\energy_for_rag.md,2,general


## Cell Guide: Generate Embeddings

This cell converts each text chunk into a dense vector representation and records the embedding dimension used by the retrieval system.


In [15]:
embedding_model = SentenceTransformer(CONFIG["EMBEDDING_MODEL_NAME"])
passages = [f"passage: {text}" for text in chunk_df["content"].tolist()]
embeddings = embedding_model.encode(
    passages,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
EMBEDDING_DIM = int(embeddings.shape[1])

print("Embedding shape:", embeddings.shape)
print("Embedding dimension:", EMBEDDING_DIM)
mark_step("embedding_generation", "passed", f"Generated embeddings with dimension {EMBEDDING_DIM}.")

Batches: 100%|██████████| 1/1 [00:22<00:00, 22.86s/it]

Embedding shape: (23, 1024)
Embedding dimension: 1024


## Cell Guide: Local Retrieval Check

This cell runs a local cosine-similarity retrieval check to verify that the embeddings are meaningful before querying Vertex AI Vector Search.


In [16]:
def local_retrieve(query: str, top_k: int = 3) -> pd.DataFrame:
    query_vector = embedding_model.encode(
        [f"query: {query}"],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )[0]
    scores = embeddings @ query_vector
    top_indices = np.argsort(-scores)[:top_k]
    result = chunk_df.iloc[top_indices].copy()
    result.insert(1, "score", scores[top_indices])
    return result[["chunk_id", "score", "content"]]


sample_query = "What assumptions are used in the household energy cost analysis?"
local_results = local_retrieve(sample_query, top_k=3)
display(local_results)
mark_step("local_sanity_retrieval", "passed", "Local cosine similarity sanity check succeeded.")

,chunk_id,score,content
0,energy_chunk_0000,0.855461,# Document: Household Energy Cost Analysis\n# ...
11,energy_chunk_0011,0.842710,## Page 4\n\nPurpose Methodology Acquisition c...
6,energy_chunk_0006,0.840878,## Page 3\n\nPurpose Methodology Acquisition c...


## Cell Guide: Export Vertex Artifacts

This cell saves the chunk table, embedding array, and line-delimited JSON file in the format expected by Vertex AI Vector Search.


In [17]:
artifact_dir = PROJECT_ROOT / "data" / "artifacts" / "vertex_vector_search"
artifact_dir.mkdir(parents=True, exist_ok=True)

chunk_csv_path = artifact_dir / "energy_chunks.csv"
embeddings_npy_path = artifact_dir / "energy_embeddings.npy"
vertex_jsonl_path = artifact_dir / "energy_embeddings_vertex.json"

chunk_df.to_csv(chunk_csv_path, index=False, encoding="utf-8")
np.save(embeddings_npy_path, embeddings)

with vertex_jsonl_path.open("w", encoding="utf-8") as handle:
    for row, vector in zip(chunk_df.to_dict(orient="records"), embeddings):
        record = {
            "id": row["chunk_id"],
            "embedding": vector.astype(float).tolist(),
            "embedding_metadata": {
                "content": row["content"],
                "source_file": row["source_file"],
                "source_path": row["source_path"],
                "char_count": row["char_count"],
                "page_number": None if pd.isna(row["page_number"]) else int(row["page_number"]),
                "section_tag": row["section_tag"],
            },
        }
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Wrote:", vertex_jsonl_path)
mark_step("vertex_export", "passed", f"Exported Vertex JSON with metadata to {vertex_jsonl_path}.")


Wrote: D:\Intern\data\artifacts\vertex_vector_search\energy_embeddings_vertex.json


## Cloud Execution Notes

- The next cells talk to GCS and Vertex AI.
- `CREATE_VERTEX_RESOURCES=true` will create an index and endpoint, which takes time and incurs cost.
- `DEPLOY_VERTEX_INDEX=true` will deploy the index to an endpoint, which is the most expensive part of this notebook.
- If you only want the Step 5 preparation artifacts, leave the flags as `false`.

## Cell Guide: Optional GCS Upload

This cell uploads the exported embedding file to Cloud Storage only when resource creation is enabled for Vertex AI.


In [18]:
def get_storage_client() -> storage.Client:
    try:
        return storage.Client(project=CONFIG["PROJECT_ID"])
    except DefaultCredentialsError as exc:
        raise RuntimeError(auth_help()) from exc


def ensure_bucket(bucket_name: str, location: str) -> storage.Bucket:
    client = get_storage_client()
    try:
        bucket = client.get_bucket(bucket_name)
        print("Using existing bucket:", bucket.name)
        return bucket
    except NotFound:
        bucket = client.bucket(bucket_name)
        bucket.location = location
        bucket = client.create_bucket(bucket)
        print("Created bucket:", bucket.name)
        return bucket


GCS_JSONL_URI = ""
GCS_PARENT_URI = ""

if CONFIG["CREATE_VERTEX_RESOURCES"]:
    bucket = ensure_bucket(CONFIG["BUCKET_NAME"], CONFIG["REGION"])
    gcs_prefix = f"step5_retriever_pipeline/{datetime.now(UTC).strftime('%Y%m%d_%H%M%S')}"
    gcs_blob_path = f"{gcs_prefix}/{vertex_jsonl_path.name}"
    blob = bucket.blob(gcs_blob_path)
    blob.upload_from_filename(vertex_jsonl_path)
    GCS_JSONL_URI = f"gs://{bucket.name}/{gcs_blob_path}"
    GCS_PARENT_URI = f"gs://{bucket.name}/{gcs_prefix}"

    print("Uploaded JSON to:", GCS_JSONL_URI)
    mark_step("gcs_upload", "passed", f"Uploaded embeddings to {GCS_JSONL_URI}.")
else:
    mark_step("gcs_upload", "skipped", "Skipped GCS upload because CREATE_VERTEX_RESOURCES=false.")

Using existing bucket: rag-bucket-xuandong
Uploaded JSON to: gs://rag-bucket-xuandong/step5_retriever_pipeline/20260514_024220/energy_embeddings_vertex.json


## Cell Guide: Load or Create Vertex Resources

This cell either loads an existing Vertex index and endpoint or creates new ones, depending on the flags and resource identifiers in `.env`.


In [19]:
vertex_index = None
vertex_endpoint = None

if CONFIG["VERTEX_INDEX_RESOURCE_NAME"]:
    vertex_index = aiplatform.MatchingEngineIndex(CONFIG["VERTEX_INDEX_RESOURCE_NAME"])
    mark_step("vertex_index", "passed", f"Loaded existing index {CONFIG['VERTEX_INDEX_RESOURCE_NAME']}.")
elif CONFIG["VERTEX_INDEX_ID"]:
    index_resource_name = (
        f"projects/{CONFIG['PROJECT_ID']}/locations/{CONFIG['REGION']}/indexes/{CONFIG['VERTEX_INDEX_ID']}"
    )
    vertex_index = aiplatform.MatchingEngineIndex(index_resource_name)
    mark_step("vertex_index", "passed", f"Loaded existing index {index_resource_name}.")
elif CONFIG["CREATE_VERTEX_RESOURCES"]:
    try:
        vertex_index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
            display_name=CONFIG["VERTEX_INDEX_DISPLAY_NAME"],
            contents_delta_uri=GCS_PARENT_URI,
            dimensions=EMBEDDING_DIM,
            approximate_neighbors_count=10,
            distance_measure_type=me_config.DistanceMeasureType.COSINE_DISTANCE,
            leaf_node_embedding_count=500,
            leaf_nodes_to_search_percent=10,
            description="Step 5 Vertex AI Vector Search index for energy_for_rag.md",
            sync=True,
        )
        mark_step("vertex_index", "passed", f"Created index {vertex_index.resource_name}.")
    except DefaultCredentialsError as exc:
        raise RuntimeError(auth_help()) from exc
    except GoogleAPICallError as exc:
        raise RuntimeError(f"Failed to create Vertex index: {exc}") from exc
else:
    mark_step("vertex_index", "skipped", "Set CREATE_VERTEX_RESOURCES=true or VERTEX_INDEX_RESOURCE_NAME to create/load an index.")

if CONFIG["VERTEX_ENDPOINT_RESOURCE_NAME"]:
    vertex_endpoint = aiplatform.MatchingEngineIndexEndpoint(CONFIG["VERTEX_ENDPOINT_RESOURCE_NAME"])
    mark_step("vertex_endpoint", "passed", f"Loaded existing endpoint {CONFIG['VERTEX_ENDPOINT_RESOURCE_NAME']}.")
elif CONFIG["VERTEX_ENDPOINT_ID"]:
    endpoint_resource_name = (
        f"projects/{CONFIG['PROJECT_ID']}/locations/{CONFIG['REGION']}/indexEndpoints/{CONFIG['VERTEX_ENDPOINT_ID']}"
    )
    vertex_endpoint = aiplatform.MatchingEngineIndexEndpoint(endpoint_resource_name)
    mark_step("vertex_endpoint", "passed", f"Loaded existing endpoint {endpoint_resource_name}.")
elif CONFIG["CREATE_VERTEX_RESOURCES"]:
    try:
        vertex_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
            display_name=CONFIG["VERTEX_ENDPOINT_DISPLAY_NAME"],
            public_endpoint_enabled=True,
            description="Step 5 Vertex AI Vector Search endpoint for energy_for_rag.md",
            sync=True,
        )
        mark_step("vertex_endpoint", "passed", f"Created endpoint {vertex_endpoint.resource_name}.")
    except DefaultCredentialsError as exc:
        raise RuntimeError(auth_help()) from exc
    except GoogleAPICallError as exc:
        raise RuntimeError(f"Failed to create Vertex endpoint: {exc}") from exc
else:
    mark_step("vertex_endpoint", "skipped", "Set CREATE_VERTEX_RESOURCES=true or VERTEX_ENDPOINT_RESOURCE_NAME to create/load an endpoint.")

print("vertex_index:", getattr(vertex_index, "resource_name", None))
print("vertex_endpoint:", getattr(vertex_endpoint, "resource_name", None))
if vertex_endpoint is not None:
    deployed_indexes = getattr(vertex_endpoint.gca_resource, "deployed_indexes", [])
    print("deployed_indexes:", [item.id for item in deployed_indexes])

Creating MatchingEngineIndex
Create MatchingEngineIndex backing LRO: projects/313902583160/locations/asia-southeast1/indexes/953113853560881152/operations/4724874968070881280
MatchingEngineIndex created. Resource name: projects/313902583160/locations/asia-southeast1/indexes/953113853560881152
To use this MatchingEngineIndex in another session:
index = aiplatform.MatchingEngineIndex('projects/313902583160/locations/asia-southeast1/indexes/953113853560881152')
Creating MatchingEngineIndexEndpoint
Create MatchingEngineIndexEndpoint backing LRO: projects/313902583160/locations/asia-southeast1/indexEndpoints/1020795391320260608/operations/8023480220143058944
MatchingEngineIndexEndpoint created. Resource name: projects/313902583160/locations/asia-southeast1/indexEndpoints/1020795391320260608
To use this MatchingEngineIndexEndpoint in another session:
index_endpoint = aiplatform.MatchingEngineIndexEndpoint('projects/313902583160/locations/asia-southeast1/indexEndpoints/1020795391320260608')
v

## Cell Guide: Deploy Index

This cell deploys the selected index to the endpoint so the service can answer nearest-neighbor queries.


In [22]:
if CONFIG["DEPLOY_VERTEX_INDEX"]:
    if vertex_index is None or vertex_endpoint is None:
        raise ValueError("Vertex index and endpoint must exist before deployment.")
    try:
        deployed_indexes = getattr(vertex_endpoint.gca_resource, "deployed_indexes", [])
        deployed_ids = {item.id for item in deployed_indexes}
        if CONFIG["DEPLOYED_INDEX_ID"] not in deployed_ids:
            vertex_endpoint.deploy_index(
                index=vertex_index,
                deployed_index_id=CONFIG["DEPLOYED_INDEX_ID"],
                sync=True,
            )
            mark_step("vertex_deployment", "passed", f"Deployed index as {CONFIG['DEPLOYED_INDEX_ID']}.")
        else:
            mark_step("vertex_deployment", "passed", f"Deployment {CONFIG['DEPLOYED_INDEX_ID']} already exists.")
    except DefaultCredentialsError as exc:
        raise RuntimeError(auth_help()) from exc
    except AlreadyExists:
        mark_step("vertex_deployment", "passed", f"Deployment {CONFIG['DEPLOYED_INDEX_ID']} already exists on the endpoint.")
    except GoogleAPICallError as exc:
        raise RuntimeError(f"Failed to deploy Vertex index: {exc}") from exc
else:
    mark_step("vertex_deployment", "skipped", "Set DEPLOY_VERTEX_INDEX=true to deploy the index.")

print(RUN_STATE["vertex_deployment"])


Deploying index MatchingEngineIndexEndpoint index_endpoint: projects/313902583160/locations/asia-southeast1/indexEndpoints/1020795391320260608


RuntimeError: Failed to deploy Vertex index: 409 There already exists a DeployedIndex with same ID "rag_energy_test_v1" deployed or being deployed at the following IndexEndpoint: projects/313902583160/locations/asia-southeast1/indexEndpoints/7725529336568086528. Please use a different ID.

## Cell Guide: Query Vertex AI Vector Search

This cell embeds a user question, sends it to Vertex AI Vector Search, and formats the returned neighbors with their metadata and source content.


In [20]:
vertex_results_df = pd.DataFrame()

if CONFIG["RUN_VERTEX_QUERY"]:
    if vertex_endpoint is None:
        raise ValueError("Vertex endpoint must exist before querying.")
    query_text = "What assumptions are used in the household energy cost analysis?"
    query_vector = embedding_model.encode(
        [f"query: {query_text}"],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )[0].astype(float).tolist()
    response = None
    last_error = None
    for attempt in range(1, 4):
        try:
            print(f"Vertex query attempt {attempt}/3")
            response = vertex_endpoint.find_neighbors(
                deployed_index_id=CONFIG["DEPLOYED_INDEX_ID"],
                queries=[query_vector],
                num_neighbors=3,
                return_full_datapoint=True,
            )
            last_error = None
            break
        except DefaultCredentialsError as exc:
            raise RuntimeError(auth_help()) from exc
        except ServiceUnavailable as exc:
            last_error = exc
            print(f"Transient Vertex serving error on attempt {attempt}: {exc}")
            if attempt < 3:
                sleep(10)
        except GoogleAPICallError as exc:
            raise RuntimeError(f"Failed to query Vertex Vector Search: {exc}") from exc

    if last_error is not None:
        raise RuntimeError(
            "Vertex query failed after 3 retries with a transient serving error. "
            "The endpoint may still be warming up, the deployment may not be healthy yet, or there may be a temporary network issue. "
            f"Last error: {last_error}"
        ) from last_error

    deployed_indexes = getattr(vertex_endpoint.gca_resource, "deployed_indexes", [])
    deployed_ids = [item.id for item in deployed_indexes]
    print("Available deployed indexes:", deployed_ids or "<none>")

    if not response:
        vertex_results_df = pd.DataFrame(
            [
                {
                    "status": "empty_response",
                    "detail": "Vertex returned no result lists. Check whether the endpoint has a deployed index and whether DEPLOYED_INDEX_ID is correct.",
                }
            ]
        )
        display(vertex_results_df)
        mark_step("vertex_query", "failed", "Vertex returned an empty response list.")
    elif not response[0]:
        vertex_results_df = pd.DataFrame(
            [
                {
                    "status": "no_neighbors",
                    "detail": "Vertex returned a query result set, but it contains no neighbors. The deployed index may be empty or the wrong deployed index ID may be in use.",
                }
            ]
        )
        display(vertex_results_df)
        mark_step("vertex_query", "failed", "Vertex returned no neighbors for the query.")
    else:
        id_to_row = chunk_df.set_index("chunk_id").to_dict(orient="index")
        rows = []
        for neighbor in response[0]:
            metadata = getattr(neighbor, "embedding_metadata", None) or {}
            local_row = id_to_row.get(neighbor.id, {})
            rows.append(
                {
                    "neighbor_id": neighbor.id,
                    "distance": getattr(neighbor, "distance", None),
                    "source_file": metadata.get("source_file") or local_row.get("source_file"),
                    "source_path": metadata.get("source_path") or local_row.get("source_path"),
                    "content": metadata.get("content") or local_row.get("content") or "Content not found in local chunk map.",
                }
            )
        vertex_results_df = pd.DataFrame(rows)
        display(vertex_results_df)
        mark_step("vertex_query", "passed", "Vertex AI Vector Search query succeeded.")
else:
    mark_step("vertex_query", "skipped", "Set RUN_VERTEX_QUERY=true after deployment to test live retrieval.")

print(RUN_STATE["vertex_query"])

Vertex query attempt 1/3
Transient Vertex serving error on attempt 1: 503 errors resolving 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443: [field:hostname lookup error:Address lookup failed for 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443 os_error: UNAVAILABLE:getaddrinfo: WSA Error (No such host is known.
 -- 11001) {grpc_status:14}]
Vertex query attempt 2/3
Transient Vertex serving error on attempt 2: 503 errors resolving 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443: [field:hostname lookup error:Address lookup failed for 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443 os_error: UNAVAILABLE:getaddrinfo: WSA Error (No such host is known.
 -- 11001) {grpc_status:14}]
Vertex query attempt 3/3
Transient Vertex serving error on attempt 3: 503 errors resolving 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443: [field:hostname lookup error:Address lookup failed for 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443 os_

RuntimeError: Vertex query failed after 3 retries with a transient serving error. The endpoint may still be warming up, the deployment may not be healthy yet, or there may be a temporary network issue. Last error: 503 errors resolving 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443: [field:hostname lookup error:Address lookup failed for 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443 os_error: UNAVAILABLE:getaddrinfo: WSA Error (No such host is known.
 -- 11001) {grpc_status:14}]

In [11]:
test_ids = [
    "energy_chunk_0000",
    "energy_chunk_0001",
    "energy_chunk_0002",
]

datapoints = vertex_endpoint.read_index_datapoints(
    deployed_index_id=CONFIG["DEPLOYED_INDEX_ID"],
    ids=test_ids,
)

print("Datapoint count:", len(datapoints))
for dp in datapoints:
    print("ID:", dp.datapoint_id)
    print("Vector dim:", len(dp.feature_vector))
    print("Metadata:", dict(dp.embedding_metadata) if dp.embedding_metadata else {})
    print("-" * 80)


Datapoint count: 3
ID: energy_chunk_0000
Vector dim: 1024
Metadata: {'source_path': 'D:\\Intern\\data\\outputs\\energy_for_rag.md', 'content': '# Document: Household Energy Cost Analysis\n# Source: AusNet\n\n## Page 1\n\nHousehold Energy\nCost Analysis\n31 Jan 2025\nAssumptions Document\n\n---', 'char_count': 140.0, 'source_file': 'energy_for_rag.md'}
--------------------------------------------------------------------------------
ID: energy_chunk_0001
Vector dim: 1024
Metadata: {'source_path': 'D:\\Intern\\data\\outputs\\energy_for_rag.md', 'content': '## Page 2\n\nPurpose Methodology Acquisition costs\nPurpose and contents\nPurpose Contents\n•\nIncorporates our current EDPR expenditure forecasts (i.e. the distribution component),\nand relies on several internal (e.g. 2023 customer segmentation) and external, publicly\n1. Purpose\navailable data sources (ECA ‘Stepping Up’ report) for other bill components and key\ninputs/ assumptions 2. Methodology &\n•\nCaptures annual household ‘run

In [16]:
query_text = "What assumptions are used in the household energy cost analysis?"
query_vector = embedding_model.encode(
    [f"query: {query_text}"],
    normalize_embeddings=True,
    convert_to_numpy=True,
)[0].astype(float).tolist()

response = vertex_endpoint.find_neighbors(
    deployed_index_id=CONFIG["DEPLOYED_INDEX_ID"],
    queries=[query_vector],
    num_neighbors=3,
    return_full_datapoint=True,
)

print("Result groups:", len(response))
for neighbors in response:
    print("Neighbors:", len(neighbors))
    for n in neighbors:
        print(n.id, n.distance, n.embedding_metadata.get("source_file"))


Result groups: 1
Neighbors: 3
energy_chunk_0000 0.144539475440979 energy_for_rag.md
energy_chunk_0011 0.15729016065597534 energy_for_rag.md
energy_chunk_0006 0.15912199020385742 energy_for_rag.md


## Cell Guide: Build the Retriever Pipeline

This cell defines reusable dense, keyword, hybrid, reranking, and metadata-filtering helpers so the notebook exposes a proper retriever pipeline instead of one-off query snippets.


In [23]:
STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "for", "with", "on", "is", "are", "be", "by",
    "what", "which", "how", "when", "where", "why", "it", "this", "that", "from", "as", "at",
}

chunk_id_to_index = {chunk_id: idx for idx, chunk_id in enumerate(chunk_df["chunk_id"].tolist())}
id_to_row = chunk_df.set_index("chunk_id").to_dict(orient="index")


def tokenize(text):
    return [token for token in re.findall(r"[a-z0-9]+", text.lower()) if token not in STOPWORDS]


def keyword_score(query, text):
    query_tokens = tokenize(query)
    if not query_tokens:
        return 0.0
    text_tokens = tokenize(text)
    if not text_tokens:
        return 0.0
    text_counts = pd.Series(text_tokens).value_counts().to_dict()
    overlap = sum(min(text_counts.get(token, 0), 1) for token in query_tokens)
    coverage = overlap / len(set(query_tokens))
    density = overlap / max(len(text_tokens), 1)
    return float((0.85 * coverage) + (0.15 * density))


def apply_metadata_filter(df, metadata_filter=None):
    filtered = df.copy()
    if not metadata_filter:
        return filtered
    for key, value in metadata_filter.items():
        if key not in filtered.columns:
            continue
        if isinstance(value, (list, tuple, set)):
            filtered = filtered[filtered[key].isin(list(value))]
        else:
            filtered = filtered[filtered[key] == value]
    return filtered


def local_dense_retrieve(query, candidate_k=8, metadata_filter=None):
    allowed_df = apply_metadata_filter(chunk_df, metadata_filter)
    if allowed_df.empty:
        return pd.DataFrame()
    query_vector = embedding_model.encode(
        [f"query: {query}"],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )[0]
    candidate_indices = [chunk_id_to_index[chunk_id] for chunk_id in allowed_df["chunk_id"].tolist()]
    candidate_embeddings = embeddings[candidate_indices]
    scores = candidate_embeddings @ query_vector
    top_positions = np.argsort(-scores)[:candidate_k]
    top_df = allowed_df.iloc[top_positions].copy()
    top_df.insert(1, "dense_score", scores[top_positions])
    top_df.insert(2, "dense_distance", 1 - scores[top_positions])
    return top_df.reset_index(drop=True)


def vertex_dense_retrieve(query, candidate_k=8, metadata_filter=None):
    if vertex_endpoint is None:
        return pd.DataFrame()
    try:
        query_vector = embedding_model.encode(
            [f"query: {query}"],
            normalize_embeddings=True,
            convert_to_numpy=True,
        )[0].astype(float).tolist()
        response = vertex_endpoint.find_neighbors(
            deployed_index_id=CONFIG["DEPLOYED_INDEX_ID"],
            queries=[query_vector],
            num_neighbors=candidate_k,
            return_full_datapoint=True,
        )
    except Exception as exc:
        print(f"Vertex dense retrieval fallback triggered: {exc}")
        return pd.DataFrame()

    if not response or not response[0]:
        return pd.DataFrame()

    rows = []
    for neighbor in response[0]:
        metadata = getattr(neighbor, "embedding_metadata", None) or {}
        local_row = id_to_row.get(neighbor.id, {})
        rows.append(
            {
                "chunk_id": neighbor.id,
                "dense_distance": getattr(neighbor, "distance", None),
                "dense_score": 1 - float(getattr(neighbor, "distance", 1.0)),
                "content": metadata.get("content") or local_row.get("content"),
                "source_file": metadata.get("source_file") or local_row.get("source_file"),
                "source_path": metadata.get("source_path") or local_row.get("source_path"),
                "char_count": metadata.get("char_count") or local_row.get("char_count"),
                "page_number": metadata.get("page_number") or local_row.get("page_number"),
                "section_tag": metadata.get("section_tag") or local_row.get("section_tag"),
            }
        )
    result = pd.DataFrame(rows)
    return apply_metadata_filter(result, metadata_filter).reset_index(drop=True)


def build_keyword_candidates(query, candidate_k=8, metadata_filter=None):
    allowed_df = apply_metadata_filter(chunk_df, metadata_filter)
    if allowed_df.empty:
        return pd.DataFrame()
    keyword_df = allowed_df.copy()
    keyword_df["keyword_score"] = keyword_df["content"].apply(lambda text: keyword_score(query, text))
    keyword_df = keyword_df.sort_values(["keyword_score", "char_count"], ascending=[False, False]).head(candidate_k)
    return keyword_df.reset_index(drop=True)


def hybrid_retrieve(
    query,
    top_k=3,
    candidate_k=8,
    metadata_filter=None,
    dense_weight=0.7,
    keyword_weight=0.3,
):
    dense_df = vertex_dense_retrieve(query, candidate_k=candidate_k, metadata_filter=metadata_filter)
    retrieval_backend = "vertex"
    if dense_df.empty:
        dense_df = local_dense_retrieve(query, candidate_k=candidate_k, metadata_filter=metadata_filter)
        retrieval_backend = "local_fallback"

    keyword_df = build_keyword_candidates(query, candidate_k=candidate_k, metadata_filter=metadata_filter)
    base_df = apply_metadata_filter(chunk_df, metadata_filter).copy()
    if base_df.empty:
        return pd.DataFrame()

    candidate_ids = set(dense_df.get("chunk_id", pd.Series(dtype=str)).tolist())
    candidate_ids.update(keyword_df.get("chunk_id", pd.Series(dtype=str)).tolist())
    combined = base_df[base_df["chunk_id"].isin(candidate_ids)].copy()
    if combined.empty:
        return pd.DataFrame()

    combined = combined.merge(
        dense_df[["chunk_id", "dense_score", "dense_distance"]],
        on="chunk_id",
        how="left",
    )
    combined = combined.merge(
        keyword_df[["chunk_id", "keyword_score"]],
        on="chunk_id",
        how="left",
    )
    combined[["dense_score", "keyword_score"]] = combined[["dense_score", "keyword_score"]].fillna(0.0)
    combined["dense_distance"] = combined["dense_distance"].fillna(1.0)
    combined["hybrid_score"] = (dense_weight * combined["dense_score"]) + (keyword_weight * combined["keyword_score"])
    combined["rerank_score"] = combined["hybrid_score"] + np.where(combined["keyword_score"] > 0, 0.02, 0.0)
    combined["retrieval_backend"] = retrieval_backend
    combined = combined.sort_values(["rerank_score", "dense_score", "keyword_score"], ascending=[False, False, False])
    return combined[
        [
            "chunk_id",
            "retrieval_backend",
            "page_number",
            "section_tag",
            "dense_score",
            "keyword_score",
            "hybrid_score",
            "rerank_score",
            "content",
        ]
    ].head(top_k).reset_index(drop=True)


mark_step("retriever_pipeline", "passed", "Built reusable dense, hybrid, reranking, and metadata-filtering retrieval helpers.")
print(RUN_STATE["retriever_pipeline"])


{'status': 'passed', 'detail': 'Built reusable dense, hybrid, reranking, and metadata-filtering retrieval helpers.', 'timestamp': '2026-05-14T02:45:15Z'}


## Cell Guide: Validate Hybrid Retrieval and Metadata Filtering

This cell validates the Step 5 retriever by running sample queries through the reusable pipeline, showing reranked top-K chunks, and demonstrating metadata-based filtering.


In [24]:
sample_queries = [
    "What assumptions are used in the household energy cost analysis?",
    "Which household types are included in the analysis?",
]

hybrid_demo_frames = []
for query in sample_queries:
    result_df = hybrid_retrieve(query=query, top_k=3, candidate_k=8)
    result_df.insert(0, "query", query)
    hybrid_demo_frames.append(result_df)

hybrid_demo_df = pd.concat(hybrid_demo_frames, ignore_index=True)
display(hybrid_demo_df)
mark_step("hybrid_retrieval", "passed", "Hybrid retriever returned reranked top-K chunks for multiple sample queries.")

filtered_demo_df = hybrid_retrieve(
    query="What assumptions are used in the household energy cost analysis?",
    top_k=3,
    candidate_k=8,
    metadata_filter={"page_number": 2},
)
display(filtered_demo_df)
if filtered_demo_df.empty:
    mark_step("metadata_filtering", "failed", "Metadata-filtered retrieval returned no chunks for page_number=2.")
else:
    mark_step("metadata_filtering", "passed", "Metadata-filtered retrieval returned top-K chunks for page_number=2.")

mark_step("reranking", "passed", "Hybrid retrieval combined dense and keyword evidence into a reranked top-K result list.")
print(RUN_STATE["hybrid_retrieval"])
print(RUN_STATE["metadata_filtering"])
print(RUN_STATE["reranking"])


Vertex dense retrieval fallback triggered: 503 errors resolving 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443: [field:hostname lookup error:Address lookup failed for 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443 os_error: UNAVAILABLE:getaddrinfo: WSA Error (No such host is known.
 -- 11001) {grpc_status:14}]
Vertex dense retrieval fallback triggered: 503 errors resolving 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443: [field:hostname lookup error:Address lookup failed for 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443 os_error: UNAVAILABLE:getaddrinfo: WSA Error (No such host is known.
 -- 11001) {grpc_status:14}]


,query,chunk_id,retrieval_backend,page_number,section_tag,dense_score,keyword_score,hybrid_score,rerank_score,content
0,What assumptions are used in the household ene...,energy_chunk_0000,local_fallback,1,assumptions,0.855461,0.750000,0.823822,0.843822,# Document: Household Energy Cost Analysis\n# ...
1,What assumptions are used in the household ene...,energy_chunk_0006,local_fallback,3,assumptions,0.840878,0.573118,0.760550,0.780550,## Page 3\n\nPurpose Methodology Acquisition c...
2,What assumptions are used in the household ene...,energy_chunk_0011,local_fallback,4,assumptions,0.842710,0.430114,0.718931,0.738931,## Page 4\n\nPurpose Methodology Acquisition c...
3,Which household types are included in the anal...,energy_chunk_0002,local_fallback,2,general,0.806498,0.435000,0.695049,0.715049,"appliances/vehicle, at cost parity)\n•\nInclud..."
4,Which household types are included in the anal...,energy_chunk_0004,local_fallback,2,assumptions,0.798687,0.430556,0.688247,0.708247,. upfront purchase costs of electrical applian...
5,Which household types are included in the anal...,energy_chunk_0000,local_fallback,1,assumptions,0.789218,0.441667,0.684953,0.704953,# Document: Household Energy Cost Analysis\n# ...


Vertex dense retrieval fallback triggered: 503 errors resolving 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443: [field:hostname lookup error:Address lookup failed for 10344135.asia-southeast1-313902583160.vdb.vertexai.goog:443 os_error: UNAVAILABLE:getaddrinfo: WSA Error (No such host is known.
 -- 11001) {grpc_status:14}]


,chunk_id,retrieval_backend,page_number,section_tag,dense_score,keyword_score,hybrid_score,rerank_score,content
0,energy_chunk_0004,local_fallback,2,assumptions,0.822349,0.433333,0.705644,0.725644,. upfront purchase costs of electrical applian...
1,energy_chunk_0001,local_fallback,2,assumptions,0.817065,0.429891,0.700913,0.720913,## Page 2\n\nPurpose Methodology Acquisition c...
2,energy_chunk_0003,local_fallback,2,assumptions,0.796187,0.433491,0.687378,0.707378,| Purpose Methodology Acquisition costs Purpos...


{'status': 'passed', 'detail': 'Hybrid retriever returned reranked top-K chunks for multiple sample queries.', 'timestamp': '2026-05-14T02:45:28Z'}
{'status': 'passed', 'detail': 'Metadata-filtered retrieval returned top-K chunks for page_number=2.', 'timestamp': '2026-05-14T02:45:28Z'}
{'status': 'passed', 'detail': 'Hybrid retrieval combined dense and keyword evidence into a reranked top-K result list.', 'timestamp': '2026-05-14T02:45:28Z'}


## Cell Guide: Summarize Test Results

This cell assembles the execution results into a compact validation table for the Step 5 retrieval experiment.


In [12]:
test_results_df = pd.DataFrame(
    [
        {"step": step, **payload}
        for step, payload in RUN_STATE.items()
    ]
)
display(test_results_df)

summary_rows = [
    {
        "test_area": "Data loading",
        "status": RUN_STATE.get("data_loaded", {}).get("status", "not_run"),
        "evidence": RUN_STATE.get("data_loaded", {}).get("detail", ""),
    },
    {
        "test_area": "Chunking + metadata enrichment",
        "status": RUN_STATE.get("chunking", {}).get("status", "not_run"),
        "evidence": RUN_STATE.get("chunking", {}).get("detail", ""),
    },
    {
        "test_area": "Embedding generation",
        "status": RUN_STATE.get("embedding_generation", {}).get("status", "not_run"),
        "evidence": RUN_STATE.get("embedding_generation", {}).get("detail", ""),
    },
    {
        "test_area": "Vertex JSON export",
        "status": RUN_STATE.get("vertex_export", {}).get("status", "not_run"),
        "evidence": RUN_STATE.get("vertex_export", {}).get("detail", ""),
    },
    {
        "test_area": "GCS upload",
        "status": RUN_STATE.get("gcs_upload", {}).get("status", "not_run"),
        "evidence": RUN_STATE.get("gcs_upload", {}).get("detail", ""),
    },
    {
        "test_area": "Vertex index creation",
        "status": RUN_STATE.get("vertex_index", {}).get("status", "not_run"),
        "evidence": RUN_STATE.get("vertex_index", {}).get("detail", ""),
    },
    {
        "test_area": "Vertex endpoint creation",
        "status": RUN_STATE.get("vertex_endpoint", {}).get("status", "not_run"),
        "evidence": RUN_STATE.get("vertex_endpoint", {}).get("detail", ""),
    },
    {
        "test_area": "Vertex deployment",
        "status": RUN_STATE.get("vertex_deployment", {}).get("status", "not_run"),
        "evidence": RUN_STATE.get("vertex_deployment", {}).get("detail", ""),
    },
    {
        "test_area": "Dense retrieval pipeline",
        "status": RUN_STATE.get("vertex_query", {}).get("status", "not_run"),
        "evidence": RUN_STATE.get("vertex_query", {}).get("detail", ""),
    },
    {
        "test_area": "Reusable retriever pipeline",
        "status": RUN_STATE.get("retriever_pipeline", {}).get("status", "not_run"),
        "evidence": RUN_STATE.get("retriever_pipeline", {}).get("detail", ""),
    },
    {
        "test_area": "Hybrid retrieval",
        "status": RUN_STATE.get("hybrid_retrieval", {}).get("status", "not_run"),
        "evidence": RUN_STATE.get("hybrid_retrieval", {}).get("detail", ""),
    },
    {
        "test_area": "Reranking",
        "status": RUN_STATE.get("reranking", {}).get("status", "not_run"),
        "evidence": RUN_STATE.get("reranking", {}).get("detail", ""),
    },
    {
        "test_area": "Metadata filtering",
        "status": RUN_STATE.get("metadata_filtering", {}).get("status", "not_run"),
        "evidence": RUN_STATE.get("metadata_filtering", {}).get("detail", ""),
    },
]
deliverable_test_df = pd.DataFrame(summary_rows)
display(deliverable_test_df)


,step,status,detail,timestamp
0,environment_loaded,passed,Environment variables loaded and Vertex AI ini...,2026-05-14T01:28:03Z
1,data_loaded,passed,Loaded D:\Intern\data\outputs\energy_for_rag.md.,2026-05-14T01:28:09Z
2,chunking,passed,Created 23 chunks from energy_for_rag.md.,2026-05-14T01:28:19Z
3,embedding_generation,passed,Generated embeddings with dimension 1024.,2026-05-14T01:28:57Z
4,local_sanity_retrieval,passed,Local cosine similarity sanity check succeeded.,2026-05-14T01:29:01Z
5,vertex_export,passed,Exported Vertex JSON to D:\Intern\data\artifac...,2026-05-14T01:29:06Z
6,gcs_upload,passed,Uploaded embeddings to gs://rag-bucket-xuandon...,2026-05-14T01:29:30Z
7,vertex_index,passed,Created index projects/313902583160/locations/...,2026-05-14T01:30:10Z
8,vertex_endpoint,passed,Created endpoint projects/313902583160/locatio...,2026-05-14T01:30:13Z
9,vertex_deployment,passed,Deployed index as rag_energy_test_v1.,2026-05-14T01:56:09Z


,test_area,status,evidence
0,Data loading,passed,Loaded D:\Intern\data\outputs\energy_for_rag.md.
1,Chunking,passed,Created 23 chunks from energy_for_rag.md.
2,Embedding generation,passed,Generated embeddings with dimension 1024.
3,Vertex JSON export,passed,Exported Vertex JSON to D:\Intern\data\artifac...
4,GCS upload,passed,Uploaded embeddings to gs://rag-bucket-xuandon...
5,Vertex index creation,passed,Created index projects/313902583160/locations/...
6,Vertex endpoint creation,passed,Created endpoint projects/313902583160/locatio...
7,Vertex deployment,passed,Deployed index as rag_energy_test_v1.
8,Vertex live query,not_run,


## Cell Guide: Compare Database Options

This cell builds a comparison table across vector database options to support technology selection for a GCP-based RAG system.


In [ ]:
decision_table_df = pd.DataFrame(
    [
        {
            "option": "Vertex AI Vector Search",
            "category": "GCP-native managed service",
            "managed": "High",
            "ops_complexity": "Medium",
            "latency_expectation": "Low at scale",
            "filtering": "Basic metadata + namespaces",
            "fit_for_project": "Best fit for production on GCP",
        },
        {
            "option": "AlloyDB AI / pgvector",
            "category": "GCP-native relational + vector",
            "managed": "Medium",
            "ops_complexity": "Medium-High",
            "latency_expectation": "Good for transactional + vector workloads",
            "filtering": "Strong SQL filtering",
            "fit_for_project": "Good if private networking is already solved",
        },
        {
            "option": "Cloud SQL pgvector",
            "category": "Managed PostgreSQL + extension",
            "managed": "Medium",
            "ops_complexity": "Medium",
            "latency_expectation": "Moderate",
            "filtering": "Strong SQL filtering",
            "fit_for_project": "Useful for smaller production footprints",
        },
        {
            "option": "MongoDB Atlas Vector Search",
            "category": "Managed document database",
            "managed": "High",
            "ops_complexity": "Medium",
            "latency_expectation": "Good",
            "filtering": "Strong JSON filtering",
            "fit_for_project": "Good if app already centers on MongoDB",
        },
        {
            "option": "Qdrant",
            "category": "Open-source vector database",
            "managed": "Low unless cloud-hosted",
            "ops_complexity": "Medium",
            "latency_expectation": "Good",
            "filtering": "Strong payload filtering",
            "fit_for_project": "Good for flexible open-source stacks",
        },
        {
            "option": "Weaviate",
            "category": "Open-source vector database",
            "managed": "Low unless cloud-hosted",
            "ops_complexity": "Medium-High",
            "latency_expectation": "Good",
            "filtering": "Strong hybrid and metadata search",
            "fit_for_project": "Good when hybrid search is a core requirement",
        },
        {
            "option": "Pinecone",
            "category": "Managed vector database",
            "managed": "High",
            "ops_complexity": "Low",
            "latency_expectation": "Low",
            "filtering": "Good metadata filtering",
            "fit_for_project": "Strong managed option outside strict GCP-native preference",
        },
        {
            "option": "Chroma",
            "category": "Local / lightweight vector store",
            "managed": "Low",
            "ops_complexity": "Low",
            "latency_expectation": "Good for small local tests",
            "filtering": "Basic metadata filtering",
            "fit_for_project": "Good for prototyping, not first-choice production backend",
        },
    ]
)
display(decision_table_df)

## Cell Guide: Capture Pros, Cons, and Recommendation

This cell summarizes the practical strengths, trade-offs, and final recommendation for using Vertex AI Vector Search in this project.


In [ ]:
pros_cons_df = pd.DataFrame(
    [
        {
            "area": "Pros",
            "detail": "Fully managed and GCP-native, so it aligns well with Vertex AI-based GenAI architectures.",
        },
        {
            "area": "Pros",
            "detail": "Designed for large-scale ANN retrieval with low-latency serving and no self-hosted vector cluster.",
        },
        {
            "area": "Pros",
            "detail": "This notebook now demonstrates dense retrieval, hybrid retrieval, reranking, and metadata-aware filtering in one retriever pipeline.",
        },
        {
            "area": "Cons",
            "detail": "Operational workflow is heavier than local vector stores because it requires GCS, index creation, endpoint deployment, and cost management.",
        },
        {
            "area": "Cons",
            "detail": "Keyword-aware hybrid retrieval is implemented in notebook logic rather than as a native Vertex keyword index.",
        },
        {
            "area": "Cons",
            "detail": "Compared with pgvector, the developer experience is less SQL-native for teams that want relational filtering and vector search in one engine.",
        },
    ]
)
display(pros_cons_df)

step5_ready = all(
    RUN_STATE.get(step, {}).get("status") == "passed"
    for step in ["vertex_query", "retriever_pipeline", "hybrid_retrieval", "reranking", "metadata_filtering"]
)
if step5_ready:
    recommendation = (
        "Recommendation: Vertex AI Vector Search is a valid Step 5 retriever backend for this project. "
        "The notebook now demonstrates dense retrieval, keyword-aware hybrid retrieval, reranking, and metadata filtering with working top-K results on sample queries."
    )
else:
    recommendation = (
        "Recommendation: keep Vertex AI Vector Search as the primary Step 5 direction, but complete the remaining retrieval validation steps before treating the pipeline as fully finished."
    )

print(recommendation)


NameError: name 'pd' is not defined